# 04_gold

Engineers ~40 features from precursor.silver.joined and writes to
precursor.gold.features. Also computes the target variable.

╔══════════════════════════════════════════════════════════════╗
║          LOOK-AHEAD BIAS PREVENTION RULES                   ║
║                                                              ║
║  1. All feature windows use rowsBetween(-n, -1)             ║
║     NEVER rowsBetween(-n, 0)                                ║
║  2. Only the target variable uses lead()                    ║
║  3. Macro features use lag() for deltas                     ║
║  4. SEC features use rolling sums of past filings only      ║
║  5. If in doubt — use one more lag                          ║
║                                                              ║
║  A single look-ahead bias invalidates all model results.    ║
╚══════════════════════════════════════════════════════════════╝

Input:  precursor.silver.joined    (~793,717 rows)
Output: precursor.gold.features    (~750,000 rows)

In [0]:
%pip install "numpy<2.0" --no-deps

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
spark.sql("""
    SELECT DISTINCT ticker 
    FROM precursor.gold.features 
    WHERE ticker = 'SPY'
    LIMIT 5
""").show()

+------+
|ticker|
+------+
+------+



In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import logging
from datetime import datetime

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("precursor.gold")

# Warmup period: longest window is 252 days (~1 year).
# Rows before MIN_DATE will have null features and are dropped in finalize_features().
WARMUP_DAYS = 63   # 3 months needed for longest meaningful window used in core features
MIN_DATE    = "2020-04-01"  # drop rows before this date (warmup period)

logger.info("Gold pipeline initialised. MIN_DATE=%s", MIN_DATE)

INFO:precursor.gold:Gold pipeline initialised. MIN_DATE=2020-04-01


## Cell 3 — Window definitions

In [0]:
# ── Window definitions ────────────────────────────────────────────────────────
#
# ALL feature windows use rowsBetween(-n, -1).
#
# Why -1 and not 0?
# rowsBetween(-n, 0) would include the CURRENT row in the lookback window.
# That means today's close would be included in today's moving average —
# look-ahead bias because we do not know today's close until market close.
# rowsBetween(-n, -1) uses only rows that are strictly BEFORE the current row,
# i.e. only historical data available at the time the feature would be computed.
#
# W_CURRENT is used ONLY for the target variable (lead) and for last-value
# operations where we need to reference the current row's position in time.

# For lag and lead operations (accesses specific offsets, not ranges)
W_LAG     = Window.partitionBy("ticker").orderBy("date")
W_CURRENT = Window.partitionBy("ticker").orderBy("date")

# Lookback windows — strictly past data only, never current row
W_5   = Window.partitionBy("ticker").orderBy("date").rowsBetween(-5,   -1)
W_10  = Window.partitionBy("ticker").orderBy("date").rowsBetween(-10,  -1)
W_21  = Window.partitionBy("ticker").orderBy("date").rowsBetween(-21,  -1)
W_63  = Window.partitionBy("ticker").orderBy("date").rowsBetween(-63,  -1)
W_252 = Window.partitionBy("ticker").orderBy("date").rowsBetween(-252, -1)

# VIX window: not partitioned by ticker because VIX is the same for all
# tickers on a given date — partitioning by ticker would wastefully recompute
# the same value 600 times per date.
W_VIX_63 = Window.orderBy("date").rowsBetween(-63, -1)

logger.info("Window definitions loaded.")

INFO:precursor.gold:Window definitions loaded.


## Cell 4 — Price and returns features

In [0]:
def add_price_features(df: DataFrame) -> DataFrame:
    """Engineer price-based and momentum features from OHLCV data.

    All features use strictly past data (rowsBetween(-n, -1) or lag).
    The current row's close is never included in any lookback window.

    Args:
        df: Spark DataFrame from precursor.silver.joined.

    Returns:
        DataFrame with price features added.
    """
    cols_before = len(df.columns)

    # ── Returns ───────────────────────────────────────────────────────────────
    # lag(close, n) gives the close n trading days ago — pure historical data.
    # Dividing by the lagged close gives the return over that period.

    # 1-day return: most recent signal of momentum direction
    df = df.withColumn(
        "return_1d",
        (F.col("close") - F.lag("close", 1).over(W_LAG)) / F.lag("close", 1).over(W_LAG),
    )

    # 5-day return: short-term momentum (1 trading week)
    df = df.withColumn(
        "return_5d",
        (F.col("close") - F.lag("close", 5).over(W_LAG)) / F.lag("close", 5).over(W_LAG),
    )

    # 21-day return: monthly momentum — one of the strongest documented factors
    df = df.withColumn(
        "return_21d",
        (F.col("close") - F.lag("close", 21).over(W_LAG)) / F.lag("close", 21).over(W_LAG),
    )

    # 63-day return: quarterly momentum
    df = df.withColumn(
        "return_63d",
        (F.col("close") - F.lag("close", 63).over(W_LAG)) / F.lag("close", 63).over(W_LAG),
    )

    # ── Volatility ────────────────────────────────────────────────────────────
    # Rolling standard deviation of daily returns.
    # return_1d must be computed first (done above).
    # rowsBetween(-21, -1): 21 past returns, no current day.
    # If we included the current row (0), we'd be using today's return
    # (which depends on today's close) to predict today's direction — leakage.

    df = df.withColumn("volatility_21d", F.stddev("return_1d").over(W_21))
    df = df.withColumn("volatility_63d", F.stddev("return_1d").over(W_63))

    # ── Volume Z-score ────────────────────────────────────────────────────────
    # How unusual is today's volume vs the past 21 trading days.
    # +1e-8 prevents division by zero for tickers with zero volume variance.
    df = df.withColumn(
        "volume_zscore_21d",
        (F.col("volume") - F.avg("volume").over(W_21)) /
        (F.stddev("volume").over(W_21) + 1e-8),
    )

    # ── 52-week high/low position ─────────────────────────────────────────────
    # W_252 covers 252 trading days back (approx 1 year), excluding today.
    # Including today's high in today's 52-week high would be look-ahead
    # because we don't know today's high until market close.
    df = df.withColumn("price_vs_52w_high", F.col("close") / F.max("high").over(W_252))
    df = df.withColumn("price_vs_52w_low",  F.col("close") / F.min("low").over(W_252))

    # ── Moving averages ───────────────────────────────────────────────────────
    # SMA over strictly past rows — today's close excluded.
    W_50  = Window.partitionBy("ticker").orderBy("date").rowsBetween(-50,  -1)

    df = df.withColumn("sma_20",  F.avg("close").over(W_21))   # 21 rows ≈ 20-day SMA
    df = df.withColumn("sma_50",  F.avg("close").over(W_50))
    df = df.withColumn("sma_200", F.avg("close").over(W_252))

    # Price relative to moving averages.
    # Values > 1.0 mean price is above the average (bullish bias).
    # +1e-8 prevents division by zero if SMA is null during warmup.
    df = df.withColumn("price_vs_sma20",  F.col("close") / (F.col("sma_20")  + 1e-8))
    df = df.withColumn("price_vs_sma50",  F.col("close") / (F.col("sma_50")  + 1e-8))
    df = df.withColumn("price_vs_sma200", F.col("close") / (F.col("sma_200") + 1e-8))

    cols_added = len(df.columns) - cols_before
    logger.info("add_price_features: added %d columns.", cols_added)
    return df

## Cell 5 — Technical indicator features

In [0]:
def add_technical_features(df: DataFrame) -> DataFrame:
    """Engineer technical analysis indicators from price and volume data.

    All indicators use strictly past data. Where true iterative computation
    (e.g. EMA, cumulative OBV) is required, SMA or rolling-sum proxies are
    used and documented.

    Args:
        df: Spark DataFrame with price features already added.

    Returns:
        DataFrame with technical features added.
    """
    cols_before = len(df.columns)

    # ── RSI-14 ────────────────────────────────────────────────────────────────
    # Relative Strength Index over 14 trading days.
    # We use rowsBetween(-14, -1): 14 past days, no current row.
    # Including the current row would mean today's gain/loss (which depends on
    # today's close) feeds into today's RSI — look-ahead bias.
    W_RSI = Window.partitionBy("ticker").orderBy("date").rowsBetween(-14, -1)

    df = df.withColumn(
        "_delta",
        F.col("close") - F.lag("close", 1).over(W_LAG),
    )
    df = df.withColumn("_gain", F.when(F.col("_delta") > 0, F.col("_delta")).otherwise(0.0))
    df = df.withColumn("_loss", F.when(F.col("_delta") < 0, F.abs(F.col("_delta"))).otherwise(0.0))
    df = df.withColumn("_avg_gain", F.avg("_gain").over(W_RSI))
    df = df.withColumn("_avg_loss", F.avg("_loss").over(W_RSI))
    df = df.withColumn("_rs", F.col("_avg_gain") / (F.col("_avg_loss") + 1e-8))
    df = df.withColumn("rsi_14", 100.0 - (100.0 / (1.0 + F.col("_rs"))))
    df = df.drop("_delta", "_gain", "_loss", "_avg_gain", "_avg_loss", "_rs")

    # ── MACD ──────────────────────────────────────────────────────────────────
    # True MACD uses exponential moving averages (EMA), which require iterative
    # computation not expressible in a single Spark window pass. We use SMA as
    # a proxy — it captures the same crossover signal with slightly less
    # responsiveness to recent price changes. This is a documented approximation
    # used in batch feature pipelines where iterative EMA is impractical.
    W_12 = Window.partitionBy("ticker").orderBy("date").rowsBetween(-12, -1)
    W_26 = Window.partitionBy("ticker").orderBy("date").rowsBetween(-26, -1)
    W_9  = Window.partitionBy("ticker").orderBy("date").rowsBetween(-9,  -1)

    df = df.withColumn("_sma_12", F.avg("close").over(W_12))
    df = df.withColumn("_sma_26", F.avg("close").over(W_26))
    df = df.withColumn("macd",    F.col("_sma_12") - F.col("_sma_26"))
    df = df.withColumn("macd_signal",    F.avg("macd").over(W_9))
    df = df.withColumn("macd_histogram", F.col("macd") - F.col("macd_signal"))
    df = df.drop("_sma_12", "_sma_26")

    # ── Bollinger Bands ───────────────────────────────────────────────────────
    # Based on sma_20 (avg close over W_21) already computed in add_price_features.
    # rowsBetween(-21, -1): 21 past closes, no current day.
    df = df.withColumn("_bb_std",    F.stddev("close").over(W_21))
    df = df.withColumn("bb_upper",   F.col("sma_20") + (2.0 * F.col("_bb_std")))
    df = df.withColumn("bb_lower",   F.col("sma_20") - (2.0 * F.col("_bb_std")))
    df = df.withColumn(
        "bb_width",
        (F.col("bb_upper") - F.col("bb_lower")) / (F.col("sma_20") + 1e-8),
    )
    df = df.withColumn(
        "bb_position",
        (F.col("close") - F.col("bb_lower")) /
        (F.col("bb_upper") - F.col("bb_lower") + 1e-8),
    )
    df = df.drop("_bb_std")

    # ── OBV momentum proxy ────────────────────────────────────────────────────
    # True On-Balance Volume is a running cumulative sum — iterative and not
    # expressible in a single Spark window. We use a rolling signed-volume sum
    # as a proxy that captures the same directional volume pressure signal over
    # a finite lookback window.
    df = df.withColumn(
        "_obv_dir",
        F.when(F.col("close") > F.lag("close", 1).over(W_LAG),  F.col("volume"))
         .when(F.col("close") < F.lag("close", 1).over(W_LAG), -F.col("volume"))
         .otherwise(0),
    )
    df = df.withColumn("obv_5d_sum",  F.sum("_obv_dir").over(W_5))
    df = df.withColumn("obv_21d_sum", F.sum("_obv_dir").over(W_21))
    df = df.drop("_obv_dir")

    # ── ATR-14 ────────────────────────────────────────────────────────────────
    # Average True Range: measures volatility using the true daily trading range.
    # True range = max(high-low, |high-prev_close|, |low-prev_close|).
    # rowsBetween(-14, -1): 14 past true ranges, no current day.
    # Including today's high/low would be look-ahead (we don't know them until EOD).
    W_ATR = Window.partitionBy("ticker").orderBy("date").rowsBetween(-14, -1)

    df = df.withColumn(
        "_true_range",
        F.greatest(
            F.col("high") - F.col("low"),
            F.abs(F.col("high") - F.lag("close", 1).over(W_LAG)),
            F.abs(F.col("low")  - F.lag("close", 1).over(W_LAG)),
        ),
    )
    df = df.withColumn("atr_14",  F.avg("_true_range").over(W_ATR))
    df = df.withColumn("atr_pct", F.col("atr_14") / (F.col("close") + 1e-8))
    df = df.drop("_true_range")

    cols_added = len(df.columns) - cols_before
    logger.info("add_technical_features: added %d columns.", cols_added)
    return df

## Cell 6 — Macro features

In [0]:
def add_macro_features(df: DataFrame) -> DataFrame:
    """Engineer macroeconomic features from FRED indicators.

    All delta features use lag() — comparing today's macro value to a past
    value. The raw FRED values themselves are not look-ahead (they are
    published data as-of that date), but changes require a reference point
    in the past.

    VIX z-score uses W_VIX_63 which is not partitioned by ticker — VIX is
    the same for all stocks on a given date so partitioning by ticker would
    redundantly compute the same result 600 times.

    Args:
        df: Spark DataFrame with price and technical features already added.

    Returns:
        DataFrame with macro features added.
    """
    cols_before = len(df.columns)

    # ── Fed Funds Rate changes ────────────────────────────────────────────────
    # Rate delta measures the direction of monetary policy.
    # Lag over W_LAG partitioned by ticker — same DFF value per date for all
    # tickers but we need the per-ticker time ordering to be consistent.
    df = df.withColumn("fed_rate_delta_21d", F.col("DFF") - F.lag("DFF", 21).over(W_LAG))
    df = df.withColumn("fed_rate_delta_63d", F.col("DFF") - F.lag("DFF", 63).over(W_LAG))

    # ── Yield curve ───────────────────────────────────────────────────────────
    # T10Y2Y < 0 = inverted yield curve, historically predicts recession.
    df = df.withColumn("yield_curve_level",     F.col("T10Y2Y"))
    df = df.withColumn(
        "yield_curve_change_21d",
        F.col("T10Y2Y") - F.lag("T10Y2Y", 21).over(W_LAG),
    )

    # ── VIX features ──────────────────────────────────────────────────────────
    # vix_level: raw fear gauge — high VIX = high market uncertainty.
    df = df.withColumn("vix_level", F.col("VIXCLS"))

    # vix_zscore_63d: how unusual is current VIX relative to past 63 trading days.
    # W_VIX_63 is NOT partitioned by ticker — VIX is identical for all tickers
    # on a given date; partitioning by ticker would compute the same z-score 600x.
    df = df.withColumn(
        "vix_zscore_63d",
        (F.col("VIXCLS") - F.avg("VIXCLS").over(W_VIX_63)) /
        (F.stddev("VIXCLS").over(W_VIX_63) + 1e-8),
    )

    # ── Inflation ─────────────────────────────────────────────────────────────
    # Month-over-month CPI change as a proxy for realized inflation momentum.
    df = df.withColumn(
        "inflation_mom",
        (F.col("CPIAUCSL") - F.lag("CPIAUCSL", 21).over(W_LAG)) /
        (F.lag("CPIAUCSL", 21).over(W_LAG) + 1e-8),
    )

    # ── Unemployment ──────────────────────────────────────────────────────────
    # Quarterly change — unemployment is a lagging indicator so 63-day delta
    # is more informative than shorter windows.
    df = df.withColumn(
        "unemployment_delta_63d",
        F.col("UNRATE") - F.lag("UNRATE", 63).over(W_LAG),
    )

    # ── Money supply growth ───────────────────────────────────────────────────
    # M2 growth rate over 63 days — captures liquidity expansion/contraction.
    df = df.withColumn(
        "m2_growth_63d",
        (F.col("M2SL") - F.lag("M2SL", 63).over(W_LAG)) /
        (F.lag("M2SL", 63).over(W_LAG) + 1e-8),
    )

    # ── Macro regime ──────────────────────────────────────────────────────────
    # Composite regime label based on yield curve inversion and VIX level.
    # Encoded as integer for model compatibility: 0=risk_off, 1=neutral, 2=risk_on.
    df = df.withColumn(
        "macro_regime",
        F.when(
            (F.col("T10Y2Y") < 0) & (F.col("VIXCLS") > 25), F.lit(0)  # risk_off
        ).when(
            (F.col("T10Y2Y") > 0) & (F.col("VIXCLS") < 20), F.lit(2)  # risk_on
        ).otherwise(F.lit(1))  # neutral
    )

    cols_added = len(df.columns) - cols_before
    logger.info("add_macro_features: added %d columns.", cols_added)
    return df

## Cell 7 — Insider trading features

In [0]:
def add_insider_features(df: DataFrame) -> DataFrame:
    """Engineer insider trading activity features from SEC Form 4 filing counts.

    filing_count is pre-aggregated per (ticker, date) in the silver table.
    We apply rolling sums over past windows to capture filing activity trends.
    All windows use rowsBetween(-n, -1) — no current day included.

    Args:
        df: Spark DataFrame with price, technical, and macro features added.

    Returns:
        DataFrame with insider features added.
    """
    cols_before = len(df.columns)

    # ── Rolling filing counts ─────────────────────────────────────────────────
    # Sum of Form 4 filings in the lookback window.
    # rowsBetween(-5, -1): last 5 trading days ≈ 1 calendar week.
    # rowsBetween(-21, -1): last 21 trading days ≈ 1 calendar month.
    # rowsBetween(-63, -1): last 63 trading days ≈ 1 calendar quarter.
    # Including the current day's filing_count would mean we know about a filing
    # that happened today — look-ahead because filings are published after market hours.
    df = df.withColumn("insider_filings_7d",  F.sum("filing_count").over(W_5))
    df = df.withColumn("insider_filings_30d", F.sum("filing_count").over(W_21))
    df = df.withColumn("insider_filings_90d", F.sum("filing_count").over(W_63))

    # ── Insider activity spike ────────────────────────────────────────────────
    # Binary flag: 1 if this week's filings are 3x above the quarterly average.
    # avg(filing_count) over W_63 = average daily filings per day over 63 days.
    # insider_filings_7d is the 5-day sum, so we compare to 5 * quarterly_daily_avg.
    df = df.withColumn(
        "insider_activity_spike",
        F.when(
            F.col("insider_filings_7d") > F.avg("filing_count").over(W_63) * 3,
            F.lit(1),
        ).otherwise(F.lit(0)),
    )

    # ── Days since last filing ────────────────────────────────────────────────
    # Find the most recent date (strictly before today) on which a filing occurred,
    # then compute the gap in days. Cap at 252 (1 trading year) for tickers with
    # no recent filings so the feature stays bounded.
    # last_value with ignorenulls=True finds the last non-null value in the window.
    df = df.withColumn(
        "_last_filing_date",
        F.last(
            F.when(F.col("filing_count") > 0, F.col("date")).otherwise(None),
            ignorenulls=True,
        ).over(W_CURRENT),
    )
    df = df.withColumn(
        "days_since_last_filing",
        F.least(
            F.datediff(F.col("date"), F.col("_last_filing_date")),
            F.lit(252),
        ),
    )
    # Fill null (no filing ever seen in history) with 252
    df = df.withColumn(
        "days_since_last_filing",
        F.coalesce(F.col("days_since_last_filing"), F.lit(252)),
    )
    df = df.drop("_last_filing_date")

    cols_added = len(df.columns) - cols_before
    logger.info("add_insider_features: added %d columns.", cols_added)
    return df

## Cell 8 — Target variable

In [0]:
def add_target_variable(df: DataFrame) -> DataFrame:
    """Add forward-looking target variables for model training.

    ╔══════════════════════════════════════════════════════════════╗
    ║  THIS IS THE ONLY PLACE WHERE FUTURE DATA IS USED.          ║
    ║  The target looks FORWARD — that is intentional.            ║
    ║  All features look BACKWARD — that prevents leakage.        ║
    ║                                                              ║
    ║  Rows where target is null = most recent rows.              ║
    ║  We do not yet know tomorrow's price for these rows.         ║
    ║  These rows are KEPT with null targets for inference.        ║
    ║  Training must filter: WHERE target_1d IS NOT NULL           ║
    ╚══════════════════════════════════════════════════════════════╝

    Args:
        df: Spark DataFrame with all feature columns added.

    Returns:
        DataFrame with target columns added.
    """
    # lead(close, n) gives the close n trading days in the future.
    # This is intentional look-ahead — it defines what we are predicting.

    # Binary direction targets: 1 = price up, 0 = price flat or down
    df = df.withColumn(
        "target_1d",
        F.when(F.lead("close", 1).over(W_CURRENT) > F.col("close"), F.lit(1))
         .when(F.lead("close", 1).over(W_CURRENT).isNotNull(),       F.lit(0))
         .otherwise(None),
    )
    df = df.withColumn(
        "target_5d",
        F.when(F.lead("close", 5).over(W_CURRENT) > F.col("close"), F.lit(1))
         .when(F.lead("close", 5).over(W_CURRENT).isNotNull(),       F.lit(0))
         .otherwise(None),
    )
    df = df.withColumn(
        "target_21d",
        F.when(F.lead("close", 21).over(W_CURRENT) > F.col("close"), F.lit(1))
         .when(F.lead("close", 21).over(W_CURRENT).isNotNull(),       F.lit(0))
         .otherwise(None),
    )

    # Regression target: raw 1-day forward return
    df = df.withColumn(
        "target_return_1d",
        (F.lead("close", 1).over(W_CURRENT) - F.col("close")) / F.col("close"),
    )

    null_targets = df.filter(F.col("target_1d").isNull()).count()
    logger.info("add_target_variable: %d rows with null target_1d (most recent dates).", null_targets)
    return df

## Cell 9 — Drop warmup rows and null features

In [0]:
def finalize_features(df: DataFrame) -> DataFrame:
    """Drop warmup rows, log null feature counts, and add pipeline metadata.

    Drops rows before MIN_DATE (2020-04-01) where long-window features
    (63-day, 252-day) are null due to insufficient history. Rows with null
    targets are kept — they are the most recent dates used for inference.

    Args:
        df: Spark DataFrame with all features and targets added.

    Returns:
        Finalized DataFrame ready for writing to gold.
    """
    before_warmup = df.count()

    # Step 1: Drop warmup rows (before MIN_DATE).
    # The first 63 trading days have null values for 63-day windows.
    # Including them in training would silently inject rows where most
    # features are null, which causes models to learn spurious patterns.
    df = df.filter(F.col("date") >= F.lit(MIN_DATE).cast("date"))
    after_warmup = df.count()
    logger.info(
        "finalize_features: dropped %d warmup rows (before %s). %d rows remaining.",
        before_warmup - after_warmup, MIN_DATE, after_warmup,
    )

    # Step 2: Log null counts per feature column.
    core_features = [
        "return_1d", "return_5d", "return_21d", "return_63d",
        "volatility_21d", "volatility_63d", "volume_zscore_21d",
        "price_vs_52w_high", "price_vs_52w_low",
        "sma_20", "sma_50", "sma_200",
        "price_vs_sma20", "price_vs_sma50", "price_vs_sma200",
        "rsi_14", "macd", "macd_signal", "macd_histogram",
        "bb_upper", "bb_lower", "bb_width", "bb_position",
        "obv_5d_sum", "obv_21d_sum", "atr_14", "atr_pct",
        "fed_rate_delta_21d", "fed_rate_delta_63d",
        "yield_curve_level", "yield_curve_change_21d",
        "vix_level", "vix_zscore_63d",
        "inflation_mom", "unemployment_delta_63d", "m2_growth_63d",
        "macro_regime",
        "insider_filings_7d", "insider_filings_30d", "insider_filings_90d",
        "insider_activity_spike", "days_since_last_filing",
    ]

    total = df.count()
    for col in core_features:
        if col in df.columns:
            null_count = df.filter(F.col(col).isNull()).count()
            null_pct   = null_count / total * 100 if total > 0 else 0
            if null_pct > 1.0:
                logger.warning(
                    "finalize_features: %.1f%% nulls in %s (%d rows).",
                    null_pct, col, null_count,
                )

    # Step 3: Drop rows where any CORE feature is null.
    # Core features must be present for training to be valid.
    # We preserve rows where only the TARGET is null (most recent dates).
    must_have = ["return_1d", "volatility_21d", "rsi_14", "macd", "vix_level", "DFF"]
    condition  = F.lit(False)
    for col in must_have:
        condition = condition | F.col(col).isNull()

    rows_with_null_core = df.filter(condition).count()
    logger.info(
        "finalize_features: dropping %d rows with null core features.",
        rows_with_null_core,
    )
    df = df.filter(~condition)

    # Step 4: Add pipeline metadata.
    df = (
        df
        .withColumn("feature_version", F.lit("v1.0"))
        .withColumn("engineered_at",   F.current_timestamp())
    )

    final_count = df.count()
    feature_count = len([c for c in df.columns if c in core_features])
    logger.info(
        "finalize_features complete: %d rows, %d core features.",
        final_count, feature_count,
    )
    return df

## Cell 10 — Write gold table

In [0]:
def write_gold(df: DataFrame) -> None:
    """Write finalized feature table to precursor.gold.features.

    Args:
        df: Finalized DataFrame from finalize_features().
    """
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("precursor.gold.features")
    )

    result       = spark.table("precursor.gold.features")
    row_count    = result.count()
    col_count    = len(result.columns)
    null_targets = {
        "target_1d":        result.filter(F.col("target_1d").isNull()).count(),
        "target_5d":        result.filter(F.col("target_5d").isNull()).count(),
        "target_21d":       result.filter(F.col("target_21d").isNull()).count(),
        "target_return_1d": result.filter(F.col("target_return_1d").isNull()).count(),
    }

    logger.info(
        "precursor.gold.features written: %d rows, %d columns.",
        row_count, col_count,
    )
    for t, n in null_targets.items():
        logger.info("  null %s: %d rows", t, n)

    display(spark.sql("""
        SELECT ticker, date, close, return_1d, rsi_14, vix_level,
               filing_count, target_1d, target_return_1d
        FROM precursor.gold.features
        WHERE ticker = 'AAPL'
        ORDER BY date DESC
        LIMIT 5
    """))

## Cell 11 — Main execution

In [0]:
_pipeline_start = datetime.now()
logger.info("=== precursor.04_gold START at %s ===", _pipeline_start.isoformat())

try:
    logger.info("Step 1: Reading silver table.")
    _df = spark.table("precursor.silver.joined")
    _input_count = _df.count()
    logger.info("silver.joined: %d rows", _input_count)

    logger.info("Step 2: Adding price features.")
    _df = add_price_features(_df)
    logger.info("After price features: %d rows", _df.count())

    logger.info("Step 3: Adding technical features.")
    _df = add_technical_features(_df)
    logger.info("After technical features: %d rows", _df.count())

    logger.info("Step 4: Adding macro features.")
    _df = add_macro_features(_df)
    logger.info("After macro features: %d rows", _df.count())

    logger.info("Step 5: Adding insider features.")
    _df = add_insider_features(_df)
    logger.info("After insider features: %d rows", _df.count())

    logger.info("Step 6: Adding target variable.")
    _df = add_target_variable(_df)
    logger.info("After target variable: %d rows", _df.count())

    logger.info("Step 7: Finalising features.")
    _df = finalize_features(_df)
    _after_warmup  = _df.count()
    _with_targets  = _df.filter(F.col("target_1d").isNotNull()).count()
    _null_targets  = _after_warmup - _with_targets
    _feature_count = len([c for c in _df.columns if c not in [
        "ticker", "date", "sector", "close", "joined_at",
        "target_1d", "target_5d", "target_21d", "target_return_1d",
        "feature_version", "engineered_at",
    ]])

    logger.info("Step 8: Writing gold table.")
    write_gold(_df)

    _elapsed = (datetime.now() - _pipeline_start).total_seconds() / 60
    logger.info("=== precursor.04_gold END — %.2f min total ===", _elapsed)

    print("=" * 60)
    print("  PRECURSOR — GOLD PIPELINE SUMMARY")
    print("=" * 60)
    print(f"  Input rows (silver)     : {_input_count:>10,}")
    print(f"  After warmup drop       : {_after_warmup:>10,}")
    print(f"  Rows with targets       : {_with_targets:>10,}")
    print(f"  Rows without targets    : {_null_targets:>10,}  (inference only)")
    print(f"  Total features          : {_feature_count:>10,}")
    print(f"  Elapsed time            : {_elapsed:>10.2f} min")
    print("=" * 60)

except Exception as exc:
    logger.error("Gold pipeline failed: %s", exc, exc_info=True)

INFO:precursor.gold:=== precursor.04_gold START at 2026-05-01T19:33:10.717300 ===
INFO:precursor.gold:Step 1: Reading silver table.
INFO:precursor.gold:silver.joined: 793717 rows
INFO:precursor.gold:Step 2: Adding price features.
INFO:precursor.gold:add_price_features: added 15 columns.
INFO:precursor.gold:After price features: 793717 rows
INFO:precursor.gold:Step 3: Adding technical features.
INFO:precursor.gold:add_technical_features: added 12 columns.
INFO:precursor.gold:After technical features: 793717 rows
INFO:precursor.gold:Step 4: Adding macro features.
/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
INFO:precursor.gold:add_macro_features: added 10 columns.
INFO:precursor.gold:After macro features: 793717 rows
INFO:precursor.gold:Step 5: Adding insider featu

ticker,date,close,return_1d,rsi_14,vix_level,filing_count,target_1d,target_return_1d
AAPL,2026-04-30,271.35,0.004367620387163663,60.6280191372876,16.89,0,null,null
AAPL,2026-04-29,270.17,-0.0019947545343724418,62.6743934583727,18.81,0,1,0.004367620387163663
AAPL,2026-04-28,270.71,0.01158402152385922,64.43035367109326,17.83,0,0,-0.0019947545343724418
AAPL,2026-04-27,267.61,-0.012727809341105248,62.00787384486016,18.02,0,1,0.01158402152385922
AAPL,2026-04-24,271.06,-0.008667666313133177,67.04302103589596,18.71,0,0,-0.012727809341105248


INFO:precursor.gold:=== precursor.04_gold END — 0.85 min total ===


  PRECURSOR — GOLD PIPELINE SUMMARY
  Input rows (silver)     :    793,717
  After warmup drop       :    762,387
  Rows with targets       :    761,778
  Rows without targets    :        609  (inference only)
  Total features          :         58
  Elapsed time            :       0.85 min


## Cell 12 — Validation

In [0]:
logger.info("=== Running gold validation ===")

_checks: list[tuple[str, bool, str]] = []

def _check(name: str, passed: bool, detail: str = "") -> None:
    status = "PASS" if passed else "FAIL"
    msg    = f"[{status}] {name}" + (f" — {detail}" if detail else "")
    logger.info(msg)
    _checks.append((name, passed, detail))


try:
    _gf       = spark.table("precursor.gold.features")
    _gf_count = _gf.count()

    # 1. Total feature count between 35 and 50
    _feature_cols = [c for c in _gf.columns if c not in [
        "ticker", "date", "sector", "close", "open", "high", "low",
        "volume", "vwap", "adjusted", "ohlc_valid", "zero_volume",
        "extreme_move", "DFF", "UNRATE", "CPIAUCSL", "T10Y2Y", "VIXCLS", "M2SL",
        "filing_count", "joined_at",
        "target_1d", "target_5d", "target_21d", "target_return_1d",
        "feature_version", "engineered_at",
    ]]
    _check(
        "Total features between 35 and 50",
        35 <= len(_feature_cols) <= 50,
        f"{len(_feature_cols)} features",
    )

    # 2. No null return_1d
    _null_ret = _gf.filter(F.col("return_1d").isNull()).count()
    _check("No null return_1d", _null_ret == 0, f"{_null_ret} nulls")

    # 3. No null rsi_14
    _null_rsi = _gf.filter(F.col("rsi_14").isNull()).count()
    _check("No null rsi_14", _null_rsi == 0, f"{_null_rsi} nulls")

    # 4. No null vix_level
    _null_vix = _gf.filter(F.col("vix_level").isNull()).count()
    _check("No null vix_level", _null_vix == 0, f"{_null_vix} nulls")

    # 5. RSI values between 0 and 100
    _bad_rsi = _gf.filter(
        (F.col("rsi_14") < 0) | (F.col("rsi_14") > 100)
    ).count()
    _check("RSI values in [0, 100]", _bad_rsi == 0, f"{_bad_rsi} out-of-range rows")

    # 6. Volatility values >= 0
    _neg_vol = _gf.filter(F.col("volatility_21d") < 0).count()
    _check("volatility_21d >= 0", _neg_vol == 0, f"{_neg_vol} negative rows")

    # 7. Target distribution: mean between 0.45 and 0.55
    _target_mean = _gf.filter(
        F.col("target_1d").isNotNull()
    ).agg(F.avg("target_1d").alias("m")).collect()[0]["m"]
    _check(
        "target_1d mean between 0.45 and 0.55",
        _target_mean is not None and 0.45 <= _target_mean <= 0.55,
        f"mean = {_target_mean:.4f}" if _target_mean is not None else "no data",
    )

    # 8. Rows with null targets < 2000
    _null_target_count = _gf.filter(F.col("target_1d").isNull()).count()
    _check(
        "Rows with null targets < 2000",
        _null_target_count < 2000,
        f"{_null_target_count} rows",
    )

    # 9. Rows with non-null targets > 700,000
    _nonnull_targets = _gf_count - _null_target_count
    _check(
        "Rows with non-null targets > 700,000",
        _nonnull_targets > 700_000,
        f"{_nonnull_targets:,} rows",
    )

    # 10. Date range starts after 2020-04-01
    _min_date = _gf.agg(F.min("date").alias("d")).collect()[0]["d"]
    _check(
        "Date range starts after 2020-04-01",
        _min_date is not None and str(_min_date) >= "2020-04-01",
        f"min date = {_min_date}",
    )

    # 11. All 11 sectors still present
    _sector_count = _gf.select("sector").distinct().count()
    _check("All 11 sectors present", _sector_count >= 11, f"{_sector_count} sectors")

    # 12. feature_version = "v1.0" for all rows
    _bad_version = _gf.filter(F.col("feature_version") != "v1.0").count()
    _check(
        "feature_version = 'v1.0' for all rows",
        _bad_version == 0,
        f"{_bad_version} rows with wrong version",
    )

except Exception as exc:
    logger.error("Validation query failed: %s", exc, exc_info=True)
    _checks.append(("Validation queries executed without error", False, str(exc)))

_passed_n = sum(1 for _, p, _ in _checks if p)
_failed_n = len(_checks) - _passed_n

print("=" * 60)
print("  GOLD VALIDATION RESULTS")
print("=" * 60)
for name, passed, detail in _checks:
    status = "PASS" if passed else "FAIL"
    line   = f"  [{status}] {name}"
    if detail:
        line += f"\n         {detail}"
    print(line)
print("-" * 60)
print(f"  {_passed_n} passed  /  {_failed_n} failed")
if _failed_n > 0:
    print("  WARNING: validation failures detected — review logs above.")
print("=" * 60)

INFO:precursor.gold:=== Running gold validation ===
INFO:precursor.gold:[PASS] Total features between 35 and 50 — 42 features
INFO:precursor.gold:[PASS] No null return_1d — 0 nulls
INFO:precursor.gold:[PASS] No null rsi_14 — 0 nulls
INFO:precursor.gold:[PASS] No null vix_level — 0 nulls
INFO:precursor.gold:[PASS] RSI values in [0, 100] — 0 out-of-range rows
INFO:precursor.gold:[PASS] volatility_21d >= 0 — 0 negative rows
INFO:precursor.gold:[PASS] target_1d mean between 0.45 and 0.55 — mean = 0.5185
INFO:precursor.gold:[PASS] Rows with null targets < 2000 — 609 rows
INFO:precursor.gold:[PASS] Rows with non-null targets > 700,000 — 761,778 rows
INFO:precursor.gold:[PASS] Date range starts after 2020-04-01 — min date = 2020-04-01
INFO:precursor.gold:[PASS] All 11 sectors present — 11 sectors
INFO:precursor.gold:[PASS] feature_version = 'v1.0' for all rows — 0 rows with wrong version


  GOLD VALIDATION RESULTS
  [PASS] Total features between 35 and 50
         42 features
  [PASS] No null return_1d
         0 nulls
  [PASS] No null rsi_14
         0 nulls
  [PASS] No null vix_level
         0 nulls
  [PASS] RSI values in [0, 100]
         0 out-of-range rows
  [PASS] volatility_21d >= 0
         0 negative rows
  [PASS] target_1d mean between 0.45 and 0.55
         mean = 0.5185
  [PASS] Rows with null targets < 2000
         609 rows
  [PASS] Rows with non-null targets > 700,000
         761,778 rows
  [PASS] Date range starts after 2020-04-01
         min date = 2020-04-01
  [PASS] All 11 sectors present
         11 sectors
  [PASS] feature_version = 'v1.0' for all rows
         0 rows with wrong version
------------------------------------------------------------
  12 passed  /  0 failed
